# FaceSim Database Explorer
This notebook inspects the `.index` (FAISS) and `.db` (SQLite) files generated by `build_index.py`.

Because I forget things and can't trust my past self to do the work properly/

In [1]:
import sqlite3
import pandas as pd
import faiss
import os

# Define paths based on your project structure
DB_PATH = "data/database/metadata.db"   # Adjust if your script named it differently
INDEX_PATH = "data/database/faces.index"

# Ensure the files exist before proceeding
print(f"DB exists: {os.path.exists(DB_PATH)}")
print(f"Index exists: {os.path.exists(INDEX_PATH)}")

DB exists: True
Index exists: True


### 1. Inspecting the SQLite Metadata Database
see what tables exist in our SQLite database, and then fetch the first few rows to understand the structure.

In [2]:
if os.path.exists(DB_PATH):
    # Connect to the SQLite database
    conn = sqlite3.connect(DB_PATH)
    
    # Check all tables available in the database
    tables_df = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
    print("Tables in DB:")
    display(tables_df)
    
    # Assuming your main table is named something like 'faces', 'metadata', or 'people'
    try:
        table_name = tables_df.iloc[0]['name']  # grab the first table automatically
        print(f"\n--- Previewing table: '{table_name}' ---")
        df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 10;", conn)
        display(df)
        
        # Check total records
        count = pd.read_sql_query(f"SELECT COUNT(*) as exact_count FROM {table_name};", conn)
        print(f"Total records in metadata db: {count.iloc[0]['exact_count']}")
    except Exception as e:
        print(f"Error reading table: {e}")
        
    conn.close()
else:
    print("Database file not found.")

Tables in DB:


,name
0,faces



--- Previewing table: 'faces' ---


,faiss_id,name,image_path
0,0,abigail_breslin,data\raw\Open Famous People Faces\abigail_bres...
1,1,adam_sandler,data\raw\Open Famous People Faces\adam_sandler...
2,2,alexis_thorpe,data\raw\Open Famous People Faces\alexis_thorp...
3,3,amanda_seyfried,data\raw\Open Famous People Faces\amanda_seyfr...
4,4,annasophia_robb,data\raw\Open Famous People Faces\annasophia_r...
5,5,anna_kendrick,data\raw\Open Famous People Faces\anna_kendric...
6,6,anthony_hopkins,data\raw\Open Famous People Faces\anthony_hopk...
7,7,barbra_streisand,data\raw\Open Famous People Faces\barbra_strei...
8,8,benedict_cumberbatch,data\raw\Open Famous People Faces\benedict_cum...
9,9,cate_blanchett,data\raw\Open Famous People Faces\cate_blanche...


Total records in metadata db: 124


### 2. Inspecting the FAISS Vector Index
load the `.index` file to ensure that the number of face embeddings matches our database rows.

In [3]:
if os.path.exists(INDEX_PATH):
    # Load the FAISS index
    index = faiss.read_index(INDEX_PATH)
    
    print("--- FAISS Index Info ---")
    print(f"Is index trained? {index.is_trained}")
    print(f"Dimensionality of embeddings (d): {index.d}")
    print(f"Total number of vectors (ntotal): {index.ntotal}")
    print(f"Index type: {type(index)}")
    
    # Optional: fetch a specific vector
    try:
        vec_id = 0
        vector = index.reconstruct(vec_id)
        print(f"\nExtracted vector {vec_id} (showing first 10 dimensions):")
        print(vector[:10])
    except Exception as e:
        print("\nNote: This specific index type doesn't support vector reconstruction natively.")
else:
    print("FAISS Index not found.")

--- FAISS Index Info ---
Is index trained? True
Dimensionality of embeddings (d): 512
Total number of vectors (ntotal): 124
Index type: <class 'faiss.swigfaiss_avx2.IndexFlatL2'>

Extracted vector 0 (showing first 10 dimensions):
[-0.10486376  0.05571589 -0.00139601  0.00904518  0.02360676  0.00235225
 -0.02481718  0.0819462   0.03011161  0.03424883]
